In [ ]:
import sys
import os

# Adds the parent directory to the python path
sys.path.append(os.path.abspath(os.path.join('..')))

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt


from datasets import load_dataset
from torch.utils.data import DataLoader


from src.data.vocabulary import Vocabulary
from src.data.dataset import IMDBDataset
from src.data.dataloader import create_dataloaders

from src.models.lstm import LSTMClassifier
from src.training.trainer import Trainer

from src.evaluation.metrics import (
    calculate_metrics,
    calculate_confusion_matrix,
)

In [ ]:
## CONFIGURATION CELL

MAX_VOCAB_SIZE = 10_000
MAX_LENGTH = 500

EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 1
DROPOUT = 0.0

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
EPOCHS = 5

In [ ]:
## reproducibility

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

In [ ]:
## load and prepare the dataset

dataset = load_dataset("stanfordnlp/imdb")

dataset

In [ ]:
# Convert the Hugging Face splits into Python lists.

train_texts_all = dataset["train"]["text"]
train_labels_all = dataset["train"]["label"]

test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

print(type(train_texts_all))
print(len(train_texts_all))
print(train_labels_all[:10])

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts_all,
    train_labels_all,
    test_size=0.20,
    random_state=SEED,
    stratify=train_labels_all,
)

print("Training samples  :", len(train_texts))
print("Validation samples:", len(val_texts))
print("Test samples      :", len(test_texts))

In [ ]:
## building the vocabulary

MAX_VOCAB_SIZE = 10_000
MAX_LENGTH = 500

vocab = Vocabulary(max_size=MAX_VOCAB_SIZE)

print("Before build:", len(vocab.word_to_id))

vocab.build(train_texts)

print("After build:", len(vocab.word_to_id))
print("First 20 words:")
print(list(vocab.word_to_id.items())[:20])

In [ ]:
# Vocabulary sanity checks

test_words = [
    "the",
    "and",
    "a",
    "i",
    "have",
    "movie",
    "bond",
    "james",
]

for word in test_words:
    print(f"{word:10s} -> {vocab.word_to_id.get(word)}")

In [ ]:
## pytorch datasets

train_dataset = IMDBDataset(
    texts=train_texts,
    labels=train_labels,
    vocabulary=vocab,
    max_length=MAX_LENGTH,
)

val_dataset = IMDBDataset(
    texts=val_texts,
    labels=val_labels,
    vocabulary=vocab,
    max_length=MAX_LENGTH,
)

test_dataset = IMDBDataset(
    texts=test_texts,
    labels=test_labels,
    vocabulary=vocab,
    max_length=MAX_LENGTH,
)

print("Train dataset:", len(train_dataset))
print("Val dataset  :", len(val_dataset))
print("Test dataset :", len(test_dataset))

In [ ]:
## data loaders

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))

In [ ]:
## data pipeline sanity check

batch_x, batch_y, batch_lengths = next(iter(train_loader))

print("Input :", batch_x.shape)
print("Labels:", batch_y.shape)
print("Lengths:", batch_lengths.shape)

print("Input dtype :", batch_x.dtype)
print("Label dtype :", batch_y.dtype)

print("Sample lengths:", batch_lengths[:10])
print("Sample labels :", batch_y[:10])

In [ ]:
# Verify padding and sequence lengths.

for i in range(5):
    length = batch_lengths[i].item()
    sequence = batch_x[i]

    real_tokens = (sequence != vocab.word_to_id["<PAD>"]).sum().item()
    padding_tokens = (sequence == vocab.word_to_id["<PAD>"]).sum().item()

    print(
        f"Sample {i}: "
        f"length={length}, "
        f"real={real_tokens}, "
        f"padding={padding_tokens}"
    )

In [ ]:
## LSTM model

lstm_model = LSTMClassifier(
    vocab_size=len(vocab.word_to_id),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    padding_idx=0,
).to(device)

print(lstm_model)

In [ ]:
# Parameter count

total_parameters = sum(
    parameter.numel()
    for parameter in lstm_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in lstm_model.parameters()
    if parameter.requires_grad
)

print("Total parameters    :", total_parameters)
print("Trainable parameters:", trainable_parameters)

In [ ]:
## forward pass sanity check

batch_x = batch_x.to(device)
batch_lengths = batch_lengths.to(device)

with torch.no_grad():
    output = lstm_model(
        batch_x,
        batch_lengths,
    )

print("Input :", batch_x.shape)
print("Output:", output.shape)

In [ ]:
### Training configuraiton, loss optimizer and trainer
criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=LEARNING_RATE,
)

trainer = Trainer(
    model=lstm_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    checkpoint_path="checkpoints/lstm/best_model.pt",
)

In [ ]:
## training

history = trainer.fit(
    epochs=EPOCHS
)

In [ ]:
## Learning curves

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(
    history["train_loss"],
    label="Train Loss",
)

plt.plot(
    history["val_loss"],
    label="Validation Loss",
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LSTM Training and Validation Loss")
plt.legend()

save_dir = "plots/lstm/"
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, "lstm_loss_curve.png"), dpi=300, bbox_inches="tight")


plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history["train_accuracy"],
    label="Train Accuracy",
)

plt.plot(
    history["val_accuracy"],
    label="Validation Accuracy",
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("LSTM Training and Validation Accuracy")
plt.legend()

save_dir = "plots/lstm/"
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, "lstm_accuracy_curve.png"), dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
## loading the best plot

checkpoint_path = "checkpoints/lstm/best_model.pt"

lstm_model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device,
    )
)

lstm_model.eval()
print("Loaded:", checkpoint_path)

In [ ]:
## Validation evaluation

val_loss, val_accuracy = trainer.validate()

print("Validation Loss    :", val_loss)
print("Validation Accuracy:", val_accuracy)

In [ ]:
y_true, y_pred, y_prob = trainer.predict(
    val_loader
)

validation_metrics = calculate_metrics(
    y_true,
    y_pred,
    y_prob,
)

print(validation_metrics)

In [ ]:
## confusion matrix

cm = calculate_confusion_matrix(
    y_true,
    y_pred,
)

print(cm)

#### same thing again

In [ ]:
## test evaluation

test_labels, test_predictions, test_probabilities = (
    trainer.predict(test_loader)
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

In [ ]:
metrics = {
    "accuracy": accuracy_score(
        test_labels,
        test_predictions,
    ),

    "precision": precision_score(
        test_labels,
        test_predictions,
    ),

    "recall": recall_score(
        test_labels,
        test_predictions,
    ),

    "f1": f1_score(
        test_labels,
        test_predictions,
    ),

    "roc_auc": roc_auc_score(
        test_labels,
        test_probabilities,
    ),
}

print(metrics)

In [ ]:
cm = confusion_matrix(
    test_labels,
    test_predictions,
)

print(cm)

In [ ]:
plt.figure(figsize=(6, 5))

plt.imshow(cm)

plt.title("LSTM Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.xticks(
    [0, 1],
    ["Negative", "Positive"],
)

plt.yticks(
    [0, 1],
    ["Negative", "Positive"],
)

plt.colorbar()

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center",
        )

save_dir = "plots/lstm/"
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, "lstm_confusion_matrix.png"), dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
## Final test set evaluation

test_y_true, test_y_pred, test_y_prob = trainer.predict(
    test_loader
)

test_metrics = calculate_metrics(
    test_y_true,
    test_y_pred,
    test_y_prob,
)

print("Test Metrics")
print("-" * 40)

for metric_name, value in test_metrics.items():
    print(f"{metric_name:12s}: {float(value):.4f}")

## Experiment Conclusions

The LSTM classifier achieved a test accuracy of 83.73%, an F1-score
of 84.18%, and a ROC-AUC of 91.24%.

Compared with the Simple RNN baseline, the LSTM produced substantial
improvements across all major evaluation metrics.

The best validation performance was obtained at Epoch 3. Although
training accuracy continued to increase during Epochs 4 and 5,
validation performance declined, indicating the beginning of
overfitting.

The experiment demonstrates that the LSTM architecture provides a
stronger baseline than the vanilla RNN for IMDB sentiment
classification, particularly in capturing information across
longer sequences.

The best validation checkpoint was therefore used for final test
evaluation.